# PAPILA Transfer Evaluation (zero-shot)

Loads each trained checkpoint from `runs/{model}_seed{seed}/best.pt`, runs it on
**all** PAPILA images (the held-out clinical set — none were used in training),
and writes, for each model, the same artifacts the trainers produce on the bundle:

* `runs/{model}_seed{seed}/papila/per_image_metrics.csv`
* `runs/{model}_seed{seed}/papila/gallery_best_worst.png`
* `runs/{model}_seed{seed}/papila/preds/` (overlay per image, if `SAVE_ALL_PREDS`)

plus a combined `runs/papila_transfer_summary.csv` ranking all checkpoints.
Designed to run headless via papermill, same as the training notebooks.

In [ ]:
MANIFEST_REL = "data/processed/manifests/combined_segmentation_manifest_with_splits.csv"
RUNS_DIR = "runs"
EVAL_SOURCE = "PAPILA"
MODELS = ["deeplabv3plus", "unet", "unetplusplus", "segformer"]
SEEDS = [41, 42, 43]
IMG_SIZE = 512          # must match the IMG_SIZE the checkpoints were trained at
NUM_CLASSES = 3
SAVE_ALL_PREDS = True
OUTPUT_SUBDIR = "papila"

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

def find_project_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "configs").exists() and (p / "src").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = find_project_root()
CLASS_NAMES = ["background", "disc", "cup"]

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"project root: {PROJECT_ROOT}")
print(f"device: {device.type}")

## Model registry — rebuilt with no pretrained download (we load trained weights)

In [ ]:
MODEL_REGISTRY = {
    "deeplabv3plus": ("DeepLabV3Plus", "resnet50"),
    "unet":          ("Unet",          "resnet50"),
    "unetplusplus":  ("UnetPlusPlus",  "resnet50"),
    "segformer":     ("Segformer",     "mit_b2"),
}

def build_model(name):
    cls_name, encoder = MODEL_REGISTRY[name]
    return getattr(smp, cls_name)(encoder_name=encoder, encoder_weights=None,
                                  in_channels=3, classes=NUM_CLASSES)

## Metrics — per-class Dice + vertical CDR (identical to the trainers)

In [ ]:
def dice_one(pred, tgt, c):
    p, t = (pred == c), (tgt == c)
    s = p.sum() + t.sum()
    return float(2 * (p & t).sum() / s) if s > 0 else float("nan")

def vertical_cdr_from_mask(mask):
    disc_rows = np.where((mask >= 1).any(axis=1))[0]
    if disc_rows.size == 0:
        return np.nan
    disc_h = disc_rows.max() - disc_rows.min() + 1
    cup_rows = np.where((mask == 2).any(axis=1))[0]
    cup_h = (cup_rows.max() - cup_rows.min() + 1) if cup_rows.size else 0
    return cup_h / disc_h

## Data — all PAPILA rows from the combined manifest

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denormalize(t):
    img = t.cpu().numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

class GlaucomaDataset(Dataset):
    def __init__(self, df, size=IMG_SIZE):
        self.df = df.reset_index(drop=True)
        self.size = size
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(PROJECT_ROOT / r["image_path"]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        msk = Image.open(PROJECT_ROOT / r["mask_path"]).resize((self.size, self.size), Image.NEAREST)
        img = (np.asarray(img, np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()
        msk = np.asarray(msk)
        if msk.ndim == 3:
            msk = msk[..., 0]
        return img, torch.from_numpy(msk.astype(np.int64))

def load_eval_df():
    df = pd.read_csv(PROJECT_ROOT / MANIFEST_REL)
    df = df[df["source_dataset"] == EVAL_SOURCE]
    return df.reset_index(drop=True)

## Overlay + per-image eval + gallery (same outputs as the trainers)

In [ ]:
cmap = np.array([[0, 0, 0], [0, 1, 0], [1, 0, 0]])  # disc=green, cup=red

def overlay_rgba(m):
    rgba = np.zeros((*m.shape, 4), np.float32)
    rgba[..., :3] = cmap[m]
    rgba[..., 3] = np.where(m > 0, 0.55, 0.0)
    return rgba

def save_overlay(path, rgb, gt, pr, title):
    fig, ax = plt.subplots(1, 3, figsize=(9, 3.2))
    ax[0].imshow(rgb); ax[0].set_title("image")
    ax[1].imshow(rgb); ax[1].imshow(overlay_rgba(gt)); ax[1].set_title("ground truth")
    ax[2].imshow(rgb); ax[2].imshow(overlay_rgba(pr)); ax[2].set_title(title)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig(path, dpi=100, bbox_inches="tight"); plt.close(fig)

@torch.no_grad()
def per_image_eval(model, df, out_dir):
    model.eval()
    ds = GlaucomaDataset(df)
    preds_dir = out_dir / "preds"
    if SAVE_ALL_PREDS:
        preds_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i in range(len(ds)):
        r = df.iloc[i]
        img, msk = ds[i]
        gt = msk.numpy()
        pr = model(img.unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
        dd, dc = dice_one(pr, gt, 1), dice_one(pr, gt, 2)
        cdr_p, cdr_g = vertical_cdr_from_mask(pr), vertical_cdr_from_mask(gt)
        cdr_err = abs((0.0 if np.isnan(cdr_p) else cdr_p) - cdr_g) if not np.isnan(cdr_g) else np.nan
        rows.append({"file_id": r["file_id"], "source": r["source_dataset"],
                     "dice_disc": dd, "dice_cup": dc,
                     "cdr_pred": cdr_p, "cdr_gt": cdr_g, "cdr_err": cdr_err})
        if SAVE_ALL_PREDS:
            save_overlay(preds_dir / f"{r['source_dataset']}_{r['file_id']}.png",
                         denormalize(img), gt, pr, f"pred (cup Dice {dc:.2f})")
    pd.DataFrame(rows).to_csv(out_dir / "per_image_metrics.csv", index=False)
    return rows

def save_gallery(model, df, rows, out_dir):
    model.eval()
    ds = GlaucomaDataset(df)
    order = sorted(range(len(rows)), key=lambda i: np.nan_to_num(rows[i]["dice_cup"], nan=-1))
    picks = order[:3] + order[-3:]
    fig, ax = plt.subplots(len(picks), 2, figsize=(7, 3.0 * len(picks)))
    for r, idx in enumerate(picks):
        img, msk = ds[idx]
        with torch.no_grad():
            pr = model(img.unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
        rgb = denormalize(img)
        ax[r, 0].imshow(rgb); ax[r, 0].imshow(overlay_rgba(msk.numpy()))
        ax[r, 0].set_title(f"{rows[idx]['source']} {rows[idx]['file_id']} — truth")
        ax[r, 1].imshow(rgb); ax[r, 1].imshow(overlay_rgba(pr))
        ax[r, 1].set_title(f"pred — cup Dice {rows[idx]['dice_cup']:.2f}")
        ax[r, 0].axis("off"); ax[r, 1].axis("off")
    plt.tight_layout()
    plt.savefig(out_dir / "gallery_best_worst.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

def _mean(rows, key):
    vals = [r[key] for r in rows if not np.isnan(r[key])]
    return float(np.mean(vals)) if vals else float("nan")

## Run every available checkpoint on PAPILA

In [ ]:
eval_df = load_eval_df()
print(f"PAPILA images to evaluate: {len(eval_df)}\n")

summary = []
for model_name in MODELS:
    for seed in SEEDS:
        run_dir = PROJECT_ROOT / RUNS_DIR / f"{model_name}_seed{seed}"
        ckpt = run_dir / "best.pt"
        if not ckpt.exists():
            print(f"skip {model_name} seed{seed}: no checkpoint at {ckpt}")
            continue
        out_dir = run_dir / OUTPUT_SUBDIR
        out_dir.mkdir(parents=True, exist_ok=True)
        model = build_model(model_name).to(device)
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state["model"])
        rows = per_image_eval(model, eval_df, out_dir)
        save_gallery(model, eval_df, rows, out_dir)
        rec = {"model": model_name, "seed": seed, "n_images": len(rows),
               "dice_disc": _mean(rows, "dice_disc"), "dice_cup": _mean(rows, "dice_cup"),
               "cdr_mae": _mean(rows, "cdr_err")}
        summary.append(rec)
        print(f"{model_name} seed{seed} | PAPILA Dice disc {rec['dice_disc']:.3f} "
              f"cup {rec['dice_cup']:.3f} | CDR MAE {rec['cdr_mae']:.3f} -> {out_dir}")

## Combined summary across all checkpoints

In [ ]:
if summary:
    sdf = pd.DataFrame(summary)
    sdf.to_csv(PROJECT_ROOT / RUNS_DIR / "papila_transfer_summary.csv", index=False)
    print("\nPAPILA transfer summary (per checkpoint):")
    print(sdf.to_string(index=False))
    print("\nMean across seeds per model:")
    print(sdf.groupby("model")[["dice_disc", "dice_cup", "cdr_mae"]].mean().to_string())
else:
    print("No checkpoints found under runs/. Train the models first.")